In [3]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter, defaultdict
from gensim.parsing.preprocessing import STOPWORDS
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.coherencemodel import CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models
import os
from bertopic import BERTopic
import math
import concurrent.futures

In [5]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
lda_model_path = '../data/models/lda/yelp_lda_model.gensim'
dictionary_path = '../data/models/lda/yelp_dictionary.dict'
bertopic_model_path = '../data/models/bertopic/yelp_bertopic_model'
stopwords_list = list(STOPWORDS) + [""]

def calculate_topic_diversity(topic_words_list):
    """
    Calcula la diversidad: (palabras únicas en todos los temas) / (total de palabras)
    """
    if not topic_words_list:
        return 0.0
    
    all_words = [word for topic in topic_words_list for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words)

def evaluate_coherence_and_diversity(input_csv, lda_path, dict_path, bertopic_path, sample_size=100000):
    lda = LdaMulticore.load(lda_path)
    dictionary = Dictionary.load(dict_path)
    bertopic_model = BERTopic.load(bertopic_path)
    
    top_n_words = 10

    lda_topics = []
    for topic_id in range(lda.num_topics):
        words = [dictionary[word_id] for word_id, _ in lda.get_topic_terms(topic_id, topn=top_n_words)]
        lda_topics.append(words)

    bertopic_topics = []
    for topic_id in bertopic_model.get_topic_info()['Topic']:
        if topic_id != -1:
            rep = bertopic_model.get_topic(topic_id)
            if rep:
                words = [word for word, _ in rep[:top_n_words]]
                bertopic_topics.append(words)

    # Usamos una muestra 
    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=sample_size, seed=42)
        .with_columns(
            pl.col('text')
            .str.replace_all(r'[^a-zA-Z\s]', '')
            .str.to_lowercase()
            .str.split(" ")
            .list.set_difference(stopwords_list)
            .alias('tokens')
        )
    )
    reference_texts = sample_df['tokens'].to_list()
    del sample_df
    
    # Diversidad
    lda_diversity = calculate_topic_diversity(lda_topics)
    bertopic_diversity = calculate_topic_diversity(bertopic_topics)

    # Coherencia C_v para LDA
    cm_lda = CoherenceModel(topics=lda_topics, texts=reference_texts, dictionary=dictionary, coherence='c_v')
    lda_coherence = cm_lda.get_coherence()

    # Coherencia C_v para BERTopic
    bertopic_dict = Dictionary(reference_texts)
    
    valid_bertopic_topics = []
    for topic in bertopic_topics:
        valid_words = [word for word in topic if word in bertopic_dict.token2id]
        
        if len(valid_words) >= 2:
            valid_bertopic_topics.append(valid_words)

    cm_bert = CoherenceModel(topics=valid_bertopic_topics, 
        texts=reference_texts, dictionary=bertopic_dict, coherence='c_v')
    
    bertopic_coherence = cm_bert.get_coherence()

    df_results = pl.DataFrame({
        "Modelo": ["LDA", "BERTopic"],
        "Coherencia (C_v)": [round(lda_coherence, 4), round(bertopic_coherence, 4)],
        "Diversidad": [round(lda_diversity, 4), round(bertopic_diversity, 4)]
    })

    print(df_results)

evaluate_coherence_and_diversity(csv_reviews, lda_model_path, dictionary_path, bertopic_model_path)
    

Loading weights: 100%|█████████████████████████████| 103/103 [00:00<00:00, 848.98it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning:

This process (pid=61338) is multi-threaded, use of fork() may lead to deadlocks in the child.

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning:

This process (pid=61338) is multi-threaded, use of fork() may lead to deadlocks in the child.

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning:

This process (pid=61338) is multi-threaded, use of fork() may lead to deadlocks in the child.

/usr/lib/python3.12/multiprocessing/popen_for

shape: (2, 3)
┌──────────────┬──────────────────┬────────────┐
│ Modelo       ┆ Coherencia (C_v) ┆ Diversidad │
│ ---          ┆ ---              ┆ ---        │
│ str          ┆ f64              ┆ f64        │
╞══════════════╪══════════════════╪════════════╡
│ LDA (Gensim) ┆ 0.5341           ┆ 0.6133     │
│ BERTopic     ┆ 0.4683           ┆ 0.607      │
└──────────────┴──────────────────┴────────────┘


In [4]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
lda_model_path = '../data/models/lda/yelp_lda_model.gensim'
dictionary_path = '../data/models/lda/yelp_dictionary.dict'
bertopic_model_path = '../data/models/bertopic/yelp_bertopic_model'
lda_html_output = '../results/visualizations/lda_intertopic_map.html'
intertopic_html_output = '../results/visualizations/bertopic_intertopic_map.html'
words_html_output = '../results/visualizations/bertopic_topic_words.html'


stopwords_list = list(STOPWORDS) + [""]

os.makedirs('../results/visualizations', exist_ok=True)

def generate_visualizations(input_csv, lda_path, dict_path, bertopic_path, sample_size_for_vis=50000):
    lda = LdaMulticore.load(lda_path)
    dictionary = Dictionary.load(dict_path)
    bertopic_model = BERTopic.load(bertopic_path)
    
    sample_df_lda = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=sample_size_for_vis, seed=42)
        .with_columns(
            pl.col('text')
            .str.replace_all(r'[^a-zA-Z\s]', '')
            .str.to_lowercase()
            .str.split(" ")
            .list.set_difference(stopwords_list)
            .alias('tokens')
        )
    )
    
    texts_tokens = sample_df_lda['tokens'].to_list()
    bow_corpus = [dictionary.doc2bow(text) for text in texts_tokens]
    del sample_df_lda

    sample_df_bert = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=sample_size_for_vis, seed=42)
    )
    docs_bert = sample_df_bert['text'].to_list()
    del sample_df_bert
    
    vis_data_lda = pyLDAvis.gensim_models.prepare(
        lda, 
        bow_corpus, 
        dictionary, 
        mds='pcoa',
        R=10
    )
    
    pyLDAvis.save_html(vis_data_lda, lda_html_output)
    print(f"Intertopic Map y Word Barchart de LDA en: {lda_html_output}")
    
    fig_intertopic = bertopic_model.visualize_topics(top_n_topics=15)
    
    fig_words = bertopic_model.visualize_barchart(top_n_topics=15, n_words=5)
        
    fig_intertopic.write_html(intertopic_html_output)
    fig_words.write_html(words_html_output)
    
    print(f"BERTopic Intertopic Map en: {intertopic_html_output}")
    print(f"BERTopic Word Barchart en: {words_html_output}")

generate_visualizations(csv_reviews, lda_model_path, dictionary_path, bertopic_model_path)

Loading weights: 100%|█████████████████████████████| 103/103 [00:00<00:00, 769.85it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2. Generando pyLDAvis para LDA... (esto puede tardar unos minutos)
 -> Guardado pyLDAvis de LDA en: ../results/visualizations/lda_intertopic_map.html
 -> Tiempo LDA-vis: 0.32 minutos.
3. Generando visualizaciones para BERTopic... (Súper rápido con tu 3060)
 -> Guardado BERTopic Intertopic Map en: ../results/visualizations/bertopic_intertopic_map.html
 -> Guardado BERTopic Word Barchart en: ../results/visualizations/bertopic_topic_words.html
 -> Tiempo BERTopic-vis: 0.21 minutos.
